## Tools for Quote Agent

In [24]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import datetime
import json

In [25]:
PRODUCT_DB = {
    "LEGO-001": {
        "product_id": "LEGO-001",
        "name": "Classic Building Blocks Set",
        "brand": "Lego",
        "category": "Educational Toys",
        "unit_price": 4.99,
        "currency": "USD",
        "description": "Standard colourful building blocks.",
        "min_order_qty": 10,
    },
    "LEGO-002": {
        "product_id": "LEGO-002",
        "name": "Technic Gear Pack",
        "brand": "Lego",
        "category": "STEM Toys",
        "unit_price": 8.49,
        "currency": "USD",
        "description": "Gears, axles, and connectors for Technic builds.",
        "min_order_qty": 5,
    },
    "LEGO-003": {
        "product_id": "LEGO-003",
        "name": "Duplo Starter Kit",
        "brand": "Lego",
        "category": "Preschool Toys",
        "unit_price": 3.25,
        "currency": "USD",
        "description": "Large toddler-safe bricks.",
        "min_order_qty": 20,
    },
}

INVENTORY_DB = {
    "LEGO-001": {
        "stock": 250,
        "reserved": 40,
        "restock_date": "2026-05-20",
        "warehouse": "Mumbai-WH1",
    },
    "LEGO-002": {
        "stock": 80,
        "reserved": 10,
        "restock_date": "2026-05-15",
        "warehouse": "Mumbai-WH1",
    },
    "LEGO-003": {
        "stock": 15,
        "reserved": 5,
        "restock_date": "2026-05-25",
        "warehouse": "Mumbai-WH2",
    },
}

DISCOUNT_RULES = [
    {"min_qty": 100, "discount_pct": 20, "label": "Bulk (100+)"},
    {"min_qty": 50, "discount_pct": 10, "label": "Volume (50-99)"},
    {"min_qty": 20, "discount_pct": 5, "label": "Standard (20-49)"},
    {"min_qty": 0, "discount_pct": 0, "label": "No discount (<20)"},
]

print("✅ Databases loaded")

✅ Databases loaded


In [26]:
from langchain.tools import tool

@tool
def get_product_info(product_id: str) -> dict:
    """Retrieve full product details (name, brand, unit price, category) for a given product ID.
    Available IDs: LEGO-001 (Classic Building Blocks), LEGO-002 (Technic Gear Pack), LEGO-003 (Duplo Starter Kit).
    """
    product = PRODUCT_DB.get(product_id.upper())
    if not product:
        return {
            "error": f"Product '{product_id}' not found.",
            "available_ids": list(PRODUCT_DB.keys()),
        }
    return {"status": "ok", "product": product}


@tool
def check_inventory(product_id: str) -> dict:
    """Check current Week 3 stock levels for a product.
    Returns available qty, reserved stock, restock date, and warehouse location."""
    inv = INVENTORY_DB.get(product_id.upper())
    if not inv:
        return {"error": f"No inventory record for '{product_id}'."}
    available = inv["stock"] - inv["reserved"]
    return {
        "status": "ok",
        "product_id": product_id.upper(),
        "total_stock": inv["stock"],
        "reserved": inv["reserved"],
        "available": available,
        "restock_date": inv["restock_date"],
        "warehouse": inv["warehouse"],
        "week": "Week 3",
        "in_stock": available > 0,
    }


@tool
def calculate_quote(product_id: str, quantity: int) -> dict:
    """Calculate a price quote with tiered discounts + 18% GST.
    Tiers: qty>=100 → 20% off, qty>=50 → 10% off, qty>=20 → 5% off, qty<20 → 0% off.
    Returns full pricing breakdown: subtotal, discount, tax, grand_total."""
    product = PRODUCT_DB.get(product_id.upper())
    if not product:
        return {"error": f"Product '{product_id}' not found."}
    if quantity <= 0:
        return {"error": "Quantity must be greater than 0."}
    if quantity < product["min_order_qty"]:
        return {"error": f"Minimum order qty is {product['min_order_qty']} units."}

    rule = next(r for r in DISCOUNT_RULES if quantity >= r["min_qty"])
    unit_price = product["unit_price"]
    subtotal = unit_price * quantity
    discount_amt = subtotal * (rule["discount_pct"] / 100)
    discounted = subtotal - discount_amt
    tax_amt = discounted * 0.18
    grand_total = discounted + tax_amt

    return {
        "status": "ok",
        "product_id": product_id.upper(),
        "product_name": product["name"],
        "quantity": quantity,
        "unit_price_usd": round(unit_price, 2),
        "subtotal_usd": round(subtotal, 2),
        "discount_tier": rule["label"],
        "discount_pct": rule["discount_pct"],
        "discount_amount_usd": round(discount_amt, 2),
        "discounted_subtotal_usd": round(discounted, 2),
        "tax_rate_pct": 18,
        "tax_amount_usd": round(tax_amt, 2),
        "grand_total_usd": round(grand_total, 2),
        "currency": "USD",
        "quote_date": datetime.datetime.now().isoformat(),
    }


TOOLS = [get_product_info, check_inventory, calculate_quote]
print("✅ Tools defined:", [t.name for t in TOOLS])

✅ Tools defined: ['get_product_info', 'check_inventory', 'calculate_quote']


In [27]:
print("=" * 55)
print("TEST A — get_product_info")
print("=" * 55)
print(json.dumps(get_product_info.invoke({"product_id": "LEGO-001"}), indent=2))
print("\n--- unknown product ---")
print(json.dumps(get_product_info.invoke({"product_id": "LEGO-999"}), indent=2))

TEST A — get_product_info
{
  "status": "ok",
  "product": {
    "product_id": "LEGO-001",
    "name": "Classic Building Blocks Set",
    "brand": "Lego",
    "category": "Educational Toys",
    "unit_price": 4.99,
    "currency": "USD",
    "description": "Standard colourful building blocks.",
    "min_order_qty": 10
  }
}

--- unknown product ---
{
  "error": "Product 'LEGO-999' not found.",
  "available_ids": [
    "LEGO-001",
    "LEGO-002",
    "LEGO-003"
  ]
}


In [28]:
print("=" * 55)
print("TEST B — check_inventory (Week 3)")
print("=" * 55)
for pid in ["LEGO-001", "LEGO-002", "LEGO-003"]:
    r = check_inventory.invoke({"product_id": pid})
    s = "✅ In Stock" if r.get("in_stock") else "❌ Out of Stock"
    print(f'  {pid}  Available: {r["available"]:>4}  {s}  [{r["warehouse"]}]')

TEST B — check_inventory (Week 3)
  LEGO-001  Available:  210  ✅ In Stock  [Mumbai-WH1]
  LEGO-002  Available:   70  ✅ In Stock  [Mumbai-WH1]
  LEGO-003  Available:   10  ✅ In Stock  [Mumbai-WH2]


In [29]:
print("=" * 55)
print("TEST C — calculate_quote (all discount tiers)")
print("=" * 55)
for qty in [10, 25, 60, 110]:
    r = calculate_quote.invoke({"product_id": "LEGO-001", "quantity": qty})
    if "error" in r:
        print(f'  qty={qty:>4} → ERROR: {r["error"]}')
    else:
        print(
            f'  qty={qty:>4} | Disc: {r["discount_pct"]:>2}% | '
            f'Total: ${r["grand_total_usd"]:.2f} | {r["discount_tier"]}'
        )

print("\n--- School project: 60 x LEGO-001 ---")
print(
    json.dumps(
        calculate_quote.invoke({"product_id": "LEGO-001", "quantity": 60}), indent=2
    )
)

TEST C — calculate_quote (all discount tiers)
  qty=  10 | Disc:  0% | Total: $58.88 | No discount (<20)
  qty=  25 | Disc:  5% | Total: $139.84 | Standard (20-49)
  qty=  60 | Disc: 10% | Total: $317.96 | Volume (50-99)
  qty= 110 | Disc: 20% | Total: $518.16 | Bulk (100+)

--- School project: 60 x LEGO-001 ---
{
  "status": "ok",
  "product_id": "LEGO-001",
  "product_name": "Classic Building Blocks Set",
  "quantity": 60,
  "unit_price_usd": 4.99,
  "subtotal_usd": 299.4,
  "discount_tier": "Volume (50-99)",
  "discount_pct": 10,
  "discount_amount_usd": 29.94,
  "discounted_subtotal_usd": 269.46,
  "tax_rate_pct": 18,
  "tax_amount_usd": 48.5,
  "grand_total_usd": 317.96,
  "currency": "USD",
  "quote_date": "2026-05-08T10:33:49.446103"
}


In [45]:
SYSTEM_PROMPT = """You are a helpful sales assistant for an educational toy company.

You have three tools:
  • get_product_info  — look up product details by ID
  • check_inventory   — check Week 3 stock levels
  • calculate_quote   — compute price with discount tiers + 18% GST

For every customer request:
1. Identify the best matching product ID.
2. Call get_product_info to confirm details.
3. Call check_inventory to verify Week 3 availability.
4. Call calculate_quote to compute the price with discounts.
5. Reply with a friendly summary AND embed a JSON invoice containing:
   invoice_number, date, customer_request,
   product (id, name, brand),
   inventory (week, available_stock, warehouse),
   pricing (unit_price, quantity, subtotal, discount_pct, discount_amount,
            tax_rate, tax_amount, grand_total, currency),
   friendly_note.
"""

agent = create_agent(
    tools=TOOLS,
    model="gpt-4o-mini",
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

print("✅ LangGraph ReAct agent ready (gpt-4o-mini)")

✅ LangGraph ReAct agent ready (gpt-4o-mini)


In [46]:
from langchain_core.messages import HumanMessage

def run_agent(user_request: str, verbose: bool = True):
    """Run the Langgraph Agent and return the parsed invoice dict"""

    if verbose:
        print(f"\n{'='*60}\nUSER: {user_request}\n{'='*60}")

    config = {"configurable": {"thread_id": "2"}}

    result = agent.invoke({'messages': [HumanMessage(content=user_request)]}, config)

    print("\n🔍 Agent execution trace:", result['messages'])

    final = result['messages'][-1].content

    if verbose:
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f'  🔧 {tc["name"]}({tc["args"]})')
        print(f"\nAGENT:\n{final[:700]}")

    try:
        clean = final.replace("```json", "").replace("```", "")
        start = clean.index("{")
        end = clean.rindex("}") + 1
        invoice = json.loads(clean[start:end])
    except (ValueError, json.JSONDecodeError):
        invoice = {"raw_response": final}
    return invoice

print("✅ run_agent() ready")

✅ run_agent() ready


In [ ]:
invoice = run_agent("I want to buy 60 units of the Classic Building Blocks Set for my school project. Can you give me a quote?")
print("\n\n📄 Parsed Invoice:\n", json.dumps(invoice, indent=2))


USER: Sorry, I needed 75 instead

🔍 Agent execution trace: [HumanMessage(content='I want to buy 60 units of the Classic Building Blocks Set for my school project. Can you give me a quote?', additional_kwargs={}, response_metadata={}, id='d17d500e-98c7-432b-95c8-ac7ba5f56379'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 423, 'total_tokens': 498, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_57133166c6', 'id': 'chatcmpl-Dd8uT6SFBC08mxrqc3e6IAQ3R2nEB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0641-6f79-79a1-abc1-ad9f124e13de-0', tool_calls=[{'name': 'get_product_info', 'args': {'product_id': 'LEG